# Train path classification model

In [1]:
from utils.device import get_device

device = get_device()

PyTorch version: 2.7.1+cu128
is cuda available: True
Using device cuda


In [2]:
from utils.available_datasets import available_datasets

dataset_choice = available_datasets["PERSEVERE"]

train_split = dataset_choice.preferred_train_split
data_dir = dataset_choice.data_dir
splits_filepath = dataset_choice.splits_filepath
ndim = dataset_choice.ndim
input_channels = dataset_choice.input_channels

In [ ]:
from image_segmentation.data import ImageDataset, FundusImageDataset, ImageDatamodule

datamodule = ImageDatamodule(
    data_dir=data_dir,
    split_file_path=splits_filepath,
    train_split_name=train_split,
)
datamodule.setup()
dataset = datamodule.dataset

stats = dataset.get_dataset_stats(ndim=ndim, input_channels=input_channels, split_name=train_split, split_indices=datamodule.train_indices)

distances_hparams = None
if isinstance(dataset, FundusImageDataset):
    dataset_res = stats["estimated_resolution_mm_per_pixel"]
    distances_hparams = dataset.get_dataset_distance_hparams()

Dataset split: Train=226, Val=56, Test=71
Loading dataset stats for split 'train' from /home/morand/afs/EVAPORE/data/PERSEVERE/image_stats.json...


/home/morand/afs/EVAPORE/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Overview of the training pipeline and it's main modules

![train_pipeline](../images/train_pipeline.png "Train pipeline diagram")

## Features Generator / Features extractor
The features generator is here a pretrained UNet (see [U-Net pretraining notebook (2)](./02_pretrain_unet.ipynb))

We load the checkpoint of the pretrained UNet and use it as a features generator for our path classification model.

We replace the last 32 to 1 layers convolutional layer of the pretrained UNet by a new one with 32 output channels, and we keep the pretrained weights for the rest of the UNet. We also set `freeze_pretrained` to False to allow fine-tuning of the pretrained UNet during the training of the path classification model.

In [4]:
from path_neural_networks.models.features_generators import FeaturesGenerator, PretrainedUnetFeaturesGenerator
from utils.other import pretty_dict_print

# Switch between using the provided pretrained UNet or the one we pretrained ourselves
use_provided_unet = False

unet_ckpt_path = dataset_choice.get_checkpoint_by_id("unet_pretrained" if use_provided_unet else "unet_pretraining", -1)
print("Using UNet checkpoint:", unet_ckpt_path)

features_generator: FeaturesGenerator = PretrainedUnetFeaturesGenerator(
    ckpt_path=unet_ckpt_path,
    device=device,
    out_channels=32,
    freeze_pretrained=False,
    skip_connection=False
)
features_generator_cfg = features_generator.as_dict()
pretty_dict_print(features_generator_cfg, init_message="Features generator configuration:")

Using UNet checkpoint: /home/morand/afs/EVAPORE/checkpoints/PERSEVERE/unet_pretraining/dice_ratio_at_095_resumed/best-checkpoint-epoch=75-val_loss=0.1676.ckpt
Features generator configuration:
{
    cls: PretrainedUnetFeaturesGenerator
    out_channels: 32
    skip_connection: False
    ckpt_path: /home/morand/afs/EVAPORE/checkpoints/PERSEVERE/unet_pretraining/dice_ratio_at_095_resumed/best-checkpoint-epoch=75-val_loss=0.1676.ckpt
    freeze_pretrained: False
}


/home/morand/afs/EVAPORE/.venv/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: DiceScore metric currently defaults to `average=micro`, but will change to`average=macro` in the v1.9 release. If you've explicitly set this parameter, you can ignore this warning.
  warnings.warn(*args, **kwargs)


## Path Sampler
For each path if the image, the path sampler will sampler a set squares patches of different sizes, for each coordinate in the path, and aggregate the features in these patches using the specified method (e.g. max pooling).

Using multiple square sizes allows to capture features at different scales, which can be beneficial for vessel segmentation where vessels can have varying widths, and because our vessels are note perfectly centered on the euclidean minimum path, so we want to capture features in a larger area around the path coordinates.

The aggregation method allows to summarize the features in the sampled patches into a single feature vector for each path coordinate, which can then be used as input to the path neural network.

At the end, we get a tensor of shape (n_features * n_scales, path_length) for each path, where n_features is the number of output channels of the features generator, n_scales is the number of different square sizes used for sampling, and path_length is the number of coordinates in the path.

![path_sampler](../images/path_features_sampling.png)

In [5]:
from path_neural_networks.models.path_samplers import *

sampling_square_sizes = [1, 3, 5]
if distances_hparams:
    sampling_square_sizes = distances_hparams["scales_sampling_sizes"]
in_channels = features_generator.out_channels
sampling_aggregation_method = SamplingMaxAggregation()

path_sampler: PathSampler = MultiScaleSquarePathSampler(
    in_channels=in_channels,
    square_sizes=sampling_square_sizes,
    ndim=ndim,
    aggregation=sampling_aggregation_method
)
path_sampler_cfg = path_sampler.as_dict()
pretty_dict_print(path_sampler_cfg, init_message="Path sampler configuration:")

Path sampler configuration:
{
    cls: MultiScaleSquarePathSampler
    in_channels: 32
    out_channels: 96
    ndim: 3
    square_sizes: [1, 3, 5]
    n_scale: 3
    aggregation: {
        cls: SamplingMaxAggregation
    }
}


## Path encoder

The path encoder is a convolutional neural network that takes as input the features sampled along the path by the path sampler, and encodes them into a fixed-size feature vector that can be used for classification.

It is composed of a series of convolutional layers, followed by a global pooling operation (here a max pooling) to aggregate the features along the path, and optionally skip connections and residual blocks to improve the flow of information and gradients through the network.

At the end, we get a feature vector of size 256 for each path, which can then be used as input to a classifier to predict the class of the path (e.g. true vessel or false positive).

![path_encoder](../images/path_encoder.png "Path encoder diagram")

In [6]:
from path_neural_networks.models.path_encoders import PathEncoder, ConvMaxPoolingPathEncoder

conv_path_residual_blocks = False
conv_path_skip_connections = False
conv_path_layers = [None, None, None]

path_encoder: PathEncoder = ConvMaxPoolingPathEncoder(
    in_channels=path_sampler.out_channels, 
    hidden_layers=conv_path_layers, 
    skip_connection=conv_path_skip_connections, 
    residual_blocks=conv_path_residual_blocks
)
path_encoder_cfg = path_encoder.as_dict()
pretty_dict_print(path_encoder_cfg, init_message="Path encoder configuration:")

Path encoder configuration:
{
    cls: ConvMaxPoolingPathEncoder
    in_channels: 96
    hidden_layers: [96, 192, 384]
    kernel_size: 3
    padding: 1
    residual_blocks: False
    skip_connection: False
    pooling_operation: {
        cls: MaxPooling
        out_channels_factor: 1
    }
    out_channels: 384
}


## Path classifier

The path classifier is a fully connected network that takes as input the feature vector produced by the path encoder for each path, and outputs a binary classification (e.g. true vessel or false positive).

It is composed of a series of fully connected layers, optionally with dropout for regularization, ReLU activations, and layer normalization to improve training stability and performance.

The number of hidden layers and their sizes can be tuned to find the best architecture for the task at hand.

In [7]:
from path_neural_networks.models.path_classifiers import PathClassifier, FCNPathClassifier

path_classifier_n_hidden_layers = 3
path_classifier_dropout = 0

path_classifier: PathClassifier = FCNPathClassifier(
    in_channels=path_encoder.out_channels,
    n_hidden_layers=path_classifier_n_hidden_layers,
    num_classes=1,
    dropout=path_classifier_dropout
)
path_classifier_cfg = path_classifier.as_dict()
pretty_dict_print(path_classifier_cfg, init_message="Path classifier configuration:")

Path classifier configuration:
{
    cls: FCNPathClassifier
    in_channels: 384
    n_hidden_layers: 3
    dropout: [0.0, 0.0, 0.0]
    num_classes: 1
}


## Loading data (images, centerlines, ground truths...)

In [8]:
from math import ceil

max_dist = 100
if distances_hparams:
    max_dist = int(ceil(distances_hparams['max_dist']))

if max_dist is None:
    centerlines_dirname = "euclidean_all_centerlines"
else:
    centerlines_dirname = f"euclidean_lt_{max_dist}_centerlines"
print("Centerlines directory name:", centerlines_dirname)

Centerlines directory name: euclidean_lt_100_centerlines


In [9]:
masked = "foreground" if "foreground" in stats.keys() else "full_image"
mean, std = stats[masked]["mean"], stats[masked]["std"]
print("Dataset stats used for normalization:")
pretty_dict_print(stats)

Dataset stats used for normalization:
{
    full_image: {
        mean: [-204.2573194988062]
        std: [423.55185884334753]
    }
}


### Adding data augmentation

In [10]:
from path_neural_networks.data.augmentations import build_train_transform, build_val_transform
from path_neural_networks.data.image_centerline_datamodule import ImageCenterlineDatamodule

val_split_ratio = 0.2
split_seed = 42
shuffle_train = True

train_transforms = build_train_transform(ndim, mean, std)
val_transforms = build_val_transform(ndim, mean, std)

datamodule = ImageCenterlineDatamodule(
    data_dir=data_dir, 
    split_file_path=splits_filepath,
    centerline_dirname=centerlines_dirname,
    train_split_name=train_split,
    val_split_ratio=val_split_ratio,
    train_transforms=train_transforms,
    val_transforms=val_transforms,
    test_transforms=val_transforms,
    seed = split_seed,
    shuffle_train = shuffle_train,
    save_resolved_split=True
)
datamodule.setup()
dataset = datamodule.dataset

Dataset split: Train=226, Val=56, Test=71


### Initialize the loss function

In [11]:
from path_neural_networks.models.losses import PathClassificationLoss, WeightedBCEWithLogitsLoss, BCEWithLogitsLoss

use_pos_weight_in_loss = True

loss_fn: PathClassificationLoss
if use_pos_weight_in_loss:
    classes_stats = dataset.get_dataset_classes_stats()
    classes_ratio = classes_stats['classes_ratio']
    loss_fn = WeightedBCEWithLogitsLoss(classes_ratio=classes_ratio)
else:
    loss_fn = BCEWithLogitsLoss()

## Initialize the model with all the previous components

In [ ]:
from path_neural_networks.models import ReducedPipelineLitModule
from path_neural_networks.utils.symmetry_enforcement import SymmetryEnforcementMode

learning_rate = 3e-4
metrics = ["accuracy", "auroc", "recall", "precision", "pr_auc"]
symmetry_enforcement_mode = SymmetryEnforcementMode.NONE

model = ReducedPipelineLitModule(
    features_generator=features_generator,
    path_sampler=path_sampler,
    path_encoder=path_encoder,
    path_classifier=path_classifier,
    path_classification_loss_fn=loss_fn,
    metrics=metrics,
    lr=learning_rate,
    symmetry_enforcement_mode=symmetry_enforcement_mode,
    inference_threshold=None,
    forward_full_volume=False,
    bbox_margin_size=2**len(features_generator.net.encoder),
    path_chunk_size=4
)
cfg = model.as_dict()
pretty_dict_print(cfg)

{
    cls: ReducedPipelineLitModule
    features_generator: {
        cls: PretrainedUnetFeaturesGenerator
        out_channels: 32
        skip_connection: False
        ckpt_path: /home/morand/afs/EVAPORE/checkpoints/PERSEVERE/unet_pretraining/dice_ratio_at_095_resumed/best-checkpoint-epoch=75-val_loss=0.1676.ckpt
        freeze_pretrained: False
    }
    path_sampler: {
        cls: MultiScaleSquarePathSampler
        in_channels: 32
        out_channels: 96
        ndim: 3
        square_sizes: [1, 3, 5]
        n_scale: 3
        aggregation: {
            cls: SamplingMaxAggregation
        }
    }
    path_encoder: {
        cls: ConvMaxPoolingPathEncoder
        in_channels: 96
        hidden_layers: [96, 192, 384]
        kernel_size: 3
        padding: 1
        residual_blocks: False
        skip_connection: False
        pooling_operation: {
            cls: MaxPooling
            out_channels_factor: 1
        }
        out_channels: 384
    }
    path_classifier: {
     

In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import CSVLogger
import torch
from pytorch_lightning import Trainer

from image_segmentation.models.callbacks import SaveConfigCallback, PlotMetricsCallback

run_dir = dataset_choice.make_run_dir(base_dir="main_model_training", run_name=None)

logger = CSVLogger(save_dir=run_dir)

callbacks = [
    ModelCheckpoint(
        dirpath=run_dir,
        monitor="val_pr_auc",
        mode="max", 
        save_top_k=1, 
        filename="best-pr-auc-checkpoint-{epoch:02d}-{val_pr_auc:.4f}"
    ),
    SaveConfigCallback(cfg, run_dir),
    PlotMetricsCallback(save_dir=run_dir, metrics=["loss", "accuracy", "auroc", "recall", "precision", "pr_auc"], every_n_epochs=1),
    EarlyStopping(
        monitor="val_pr_auc",
        patience=20,
        min_delta=1e-3,
        verbose=True,
        mode="max"
    ),
]

trainer = Trainer(accelerator='gpu', 
                  devices="auto",
                  num_nodes=1,
                  max_epochs=100,
                  precision="16-mixed",
                  detect_anomaly=False, 
                  callbacks=callbacks,
                  logger=logger,
                  val_check_interval=0.1
)
torch.set_float32_matmul_precision("medium")

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [ ]:
trainer.fit(model, datamodule=datamodule)

/home/morand/afs/EVAPORE/.venv/lib/python3.12/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /home/morand/afs/EVAPORE/checkpoints/PERSEVERE/main_model_pretraining/2026-09-08_13:50:29 exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Dataset split: Train=226, Val=56, Test=71


/home/morand/afs/EVAPORE/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                        ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ features_generator          │ PretrainedUnetFeaturesGenerator │ 98.2 M │ train │     0 │
│ 1 │ path_sampler                │ MultiScaleSquarePathSampler     │      0 │ train │     0 │
│ 2 │ path_encoder                │ ConvMaxPoolingPathEncoder       │  832 K │ train │     0 │
│ 3 │ path_classifier             │ FCNPathClassifier               │ 97.8 K │ train │     0 │
│ 4 │ edge_classification_loss_fn │ WeightedBCEWithLogitsLoss       │      0 │ train │     0 │
│ 5 │ train_metrics               │ ModuleDict                      │      0 │ train │     0 │
│ 6 │ val_metrics                 │ ModuleDict                      │      0 │ train │     0 │
│ 7 │ test_metrics                │ ModuleDict                      │      0 │ train │     0 │
└───┴─────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 99.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 99.2 M                                                                                               
Total estimated model params size (MB): 396.645                                                                    
Modules in train mode: 170                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/home/morand/afs/EVAPORE/.venv/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/home/morand/afs/EVAPORE/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:
434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of 
the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.

/home/morand/afs/EVAPORE/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:
434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of 
the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.

Metric val_pr_auc improved. New best score: 0.940
Metric val_pr_auc improved by 0.023 >= min_delta = 0.001. New best score: 0.963
Metric val_pr_auc improved by 0.013 >= min_delta = 0.001. New best score: 0.976
